# 02 — Graph analysis: benign vs attack flows

Nodes represent hosts and directed edges represent flows. Each ordered one-hour window is kept as a separate graph. Flow features are edge attributes; node features are incident-edge aggregates. This notebook inspects the test-window graphs and the model-flagged subgraph.

In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import Image, display
from src.data import load_flows, temporal_window_indices
from src.preprocess import FeaturePreprocessor
from src.build_graph import build_graph_windows

cfg = yaml.safe_load(Path('configs/experiment.yaml').read_text())
flows = load_flows(cfg['data']['path'], sample=cfg['data'].get('sample'), seed=cfg['data'].get('sample_seed',17))
window_seconds = cfg['data'].get('window_seconds', 3600)
train_idx, val_idx, test_idx = temporal_window_indices(flows, window_seconds, cfg['split']['train_fraction'], cfg['split']['validation_fraction'])
prep = FeaturePreprocessor(engineered=True).fit(flows.iloc[train_idx])
X = prep.transform(flows)
test_frame = flows.iloc[test_idx].reset_index(drop=True)
test_graphs = build_graph_windows(test_frame, X[test_idx], cfg['data']['window_size'], window_seconds)
print(f'{len(test_graphs)} test graph windows')

## Graph measures

For each class-specific induced subgraph, compare degree distributions, fan-out (unique destinations per source), connected components and density. These are descriptive test-set graph statistics, not evidence of generalization beyond this dataset.

In [ ]:
def class_graph_stats(graph, label):
    ids = np.flatnonzero(graph.y.numpy() == label)
    G = nx.DiGraph()
    G.add_edges_from((graph.src_hosts[i], graph.dst_hosts[i]) for i in ids)
    if not G:
        return None
    source_fanouts=[degree for _,degree in G.out_degree() if degree>0]
    return {'flows':len(ids), 'nodes':G.number_of_nodes(), 'edges':G.number_of_edges(),
            'mean_fanout':float(np.mean(source_fanouts)),
            'weak_components':nx.number_weakly_connected_components(G), 'density':nx.density(G)}
rows=[]
for w, graph in enumerate(test_graphs):
    for y, name in [(0,'Benign'),(1,'Attack')]:
        stats=class_graph_stats(graph,y)
        if stats: rows.append({'window':w,'class':name,**stats})
stats=pd.DataFrame(rows)
display(stats.groupby('class').mean(numeric_only=True).T)
fig, axes = plt.subplots(1,2,figsize=(11,4))
for y,name,color in [(0,'Benign','#277da1'),(1,'Attack','#f94144')]:
    deg=[]; fan=[]
    for graph in test_graphs:
        ids=np.flatnonzero(graph.y.numpy()==y)
        G=nx.DiGraph(); G.add_edges_from((graph.src_hosts[i],graph.dst_hosts[i]) for i in ids)
        deg.extend(dict(G.degree()).values()); fan.extend(v for _,v in G.out_degree() if v>0)
    if deg: axes[0].hist(deg,bins=20,density=True,alpha=.55,label=name,color=color)
    if fan: axes[1].hist(fan,bins=20,density=True,alpha=.55,label=name,color=color)
axes[0].set(title='Degree distribution',xlabel='Degree',ylabel='Density')
axes[1].set(title='Unique destinations per source',xlabel='Fan-out',ylabel='Density')
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout()

## Extracted attack subgraph

The experiment script selects a test window and shows the directed edges whose E-GraphSAGE attack probability is at least 0.5. A dense high-fan-out source can be consistent with scanning, but that interpretation is not a ground-truth attack-path claim.

In [ ]:
image_path = Path('results/attack_subgraph.png')
if image_path.exists():
    display(Image(filename=str(image_path)))
else:
    print('Run python -m src.train --config configs/experiment.yaml to generate the flagged subgraph.')

## From this project to lateral movement

For lateral-movement detection, replace flow edges with timestamped authentication or host-to-host events. Keep hosts as nodes, and encode event type, account role, result, and time gap as edge attributes. A temporal model can score events within short windows, while an event store links unusual transitions across windows into candidate paths. On a provenance graph, process, file, user, and host entities become typed nodes and system events become typed edges. Evaluation then needs path-level labels and careful separation by time and enterprise, not only per-edge flow labels. Public LANL authentication data could support a separate follow-up, subject to its own access and label limitations.